# Advanced Python for Lead Engineers

## Purpose of This Notebook

This notebook covers **advanced Python topics** expected from Senior/Lead engineers:

| Topic | What You'll Learn |
|-------|-------------------|
| Decorators | Add logging, timing, retry to any function |
| Custom Exceptions | Build professional error hierarchies |
| Context Managers | Resource management (files, DB, locks) |
| Protocols | Duck typing with type hints |
| Testing | Pytest fixtures, mocking, parametrize |
| Type Hints | Generic types, TypeVar, Literal |

---

In [ ]:
# IMPORTS
import functools
import time
from typing import Callable, Any, List, Dict, Optional, TypeVar, Protocol
from abc import ABC, abstractmethod
from contextlib import contextmanager
from unittest.mock import Mock, patch

---
## 1. Decorators - The Basics

### What is a Decorator?
- A function that **wraps another function**
- Adds behavior **without modifying original code**
- Uses `@decorator_name` syntax

### How It Works:
```python
@timer
def my_func():
    pass

# Is equivalent to:
my_func = timer(my_func)
```

In [ ]:
def timer(func: Callable) -> Callable:
    """
    DECORATOR: Measures execution time of any function.
    
    HOW IT WORKS:
    1. timer() receives the original function
    2. Returns a new 'wrapper' function
    3. wrapper() calls original function + adds timing
    
    @functools.wraps preserves:
    - Function name (func.__name__)
    - Docstring (func.__doc__)
    - Other metadata
    """
    @functools.wraps(func)  # Preserve metadata
    def wrapper(*args, **kwargs):
        # BEFORE: Record start time
        start = time.time()
        
        # CALL: Execute original function
        result = func(*args, **kwargs)
        
        # AFTER: Calculate and print elapsed time
        elapsed = time.time() - start
        print(f"⏱️ {func.__name__}() took {elapsed:.4f} seconds")
        
        return result  # Return original function's result
    
    return wrapper  # Return the wrapper function

In [ ]:
@timer  # Applies timer decorator
def slow_operation(n: int) -> int:
    """Simulates a slow database query."""
    time.sleep(0.1)  # Simulate work
    return n * 2

# DEMO
print("Calling decorated function:")
result = slow_operation(5)
print(f"Result: {result}")

# Metadata is preserved thanks to @functools.wraps
print(f"\nFunction name: {slow_operation.__name__}")
print(f"Docstring: {slow_operation.__doc__}")

---
## 2. Decorators with Arguments

### The Pattern
When decorator needs arguments, we need **3 levels of nesting**:
1. **Decorator factory**: Takes arguments, returns decorator
2. **Decorator**: Takes function, returns wrapper
3. **Wrapper**: Calls original function with added behavior

In [ ]:
def retry(max_attempts: int = 3, delay: float = 0.5):
    """
    DECORATOR FACTORY: Returns a decorator with configurable retry.
    
    Usage:
        @retry(max_attempts=5, delay=1.0)
        def flaky_function():
            ...
    """
    # LEVEL 1: Factory - receives arguments
    def decorator(func: Callable) -> Callable:
        # LEVEL 2: Decorator - receives function
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # LEVEL 3: Wrapper - executes function with retry
            last_error = None
            
            for attempt in range(1, max_attempts + 1):
                try:
                    # Try to execute the function
                    return func(*args, **kwargs)
                    
                except Exception as e:
                    last_error = e
                    print(f"   ⚠️ Attempt {attempt}/{max_attempts} failed: {e}")
                    
                    if attempt < max_attempts:
                        print(f"   ⏳ Waiting {delay}s before retry...")
                        time.sleep(delay)
            
            # All attempts failed
            print(f"   ❌ All {max_attempts} attempts exhausted!")
            raise last_error
            
        return wrapper
    return decorator

In [ ]:
import random

@retry(max_attempts=3, delay=0.1)
def call_unreliable_api():
    """Simulates an API that fails 70% of the time."""
    if random.random() < 0.7:
        raise ConnectionError("API server unavailable")
    return {"status": "success", "data": [1, 2, 3]}

# DEMO: May succeed or fail depending on random
print("Calling unreliable API with retry decorator:")
try:
    result = call_unreliable_api()
    print(f"✅ Success: {result}")
except ConnectionError:
    print("❌ Failed after all retries")

---
## 3. Custom Exception Hierarchy

### Why Custom Exceptions?
- **Specific handling**: Catch `DatabaseError` vs generic `Exception`
- **Error context**: Include relevant information
- **Hierarchy**: Group related errors under parent class

In [ ]:
# EXCEPTION HIERARCHY for ETL Pipeline

class PipelineError(Exception):
    """
    BASE EXCEPTION for all pipeline errors.
    Catch this to handle ANY pipeline error.
    """
    def __init__(self, message: str, step: str = None):
        self.step = step  # Which step failed
        super().__init__(f"[{step}] {message}" if step else message)


class ExtractionError(PipelineError):
    """Failed to extract data from source."""
    def __init__(self, source: str, reason: str):
        self.source = source
        super().__init__(f"Failed to extract from {source}: {reason}", step="EXTRACT")


class TransformationError(PipelineError):
    """Failed to transform data."""
    def __init__(self, reason: str, row: dict = None):
        self.bad_row = row  # Include problematic row for debugging
        super().__init__(reason, step="TRANSFORM")


class LoadError(PipelineError):
    """Failed to load data to destination."""
    def __init__(self, destination: str, reason: str):
        self.destination = destination
        super().__init__(f"Failed to load to {destination}: {reason}", step="LOAD")

In [ ]:
def run_etl_pipeline():
    """Demonstrates exception handling in ETL."""
    
    # Simulate extraction error
    raise ExtractionError(
        source="s3://bucket/data.csv",
        reason="Access denied - check IAM permissions"
    )

# DEMO: Catching specific vs general exceptions
print("Exception handling demo:")
print("=" * 50)

try:
    run_etl_pipeline()
    
except ExtractionError as e:
    # Specific handling for extraction
    print(f"❌ Extraction failed!")
    print(f"   Source: {e.source}")
    print(f"   Error: {e}")
    
except TransformationError as e:
    # Specific handling for transformation
    print(f"❌ Transformation failed!")
    print(f"   Bad row: {e.bad_row}")
    
except PipelineError as e:
    # Catch-all for any pipeline error
    print(f"❌ Pipeline error in step: {e.step}")
    print(f"   Error: {e}")

---
## 4. Context Managers - Resource Management

### What is a Context Manager?
- Manages setup and cleanup of resources
- Ensures cleanup happens even if exception occurs
- Uses `with` statement

### Use Cases:
- Files: `with open(...) as f:`
- Database connections
- Locks for threading
- Temporary state changes

In [ ]:
# CLASS-BASED CONTEXT MANAGER

class Timer:
    """
    Context manager for timing code blocks.
    
    REQUIRED METHODS:
    - __enter__: Called when entering 'with' block
    - __exit__: Called when exiting 'with' block (even if exception)
    """
    
    def __enter__(self):
        """
        Called at start of 'with' block.
        Return value is assigned to 'as' variable.
        """
        print("⏱️ Timer started")
        self.start = time.time()
        return self  # 'as t' will be this Timer instance
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        Called at end of 'with' block.
        
        Args:
            exc_type: Exception type (or None if no exception)
            exc_val: Exception value
            exc_tb: Exception traceback
        
        Returns:
            True to suppress exception, False to propagate it
        """
        self.elapsed = time.time() - self.start
        print(f"⏱️ Timer stopped. Elapsed: {self.elapsed:.4f}s")
        
        if exc_type:
            print(f"   ⚠️ Exception occurred: {exc_val}")
        
        return False  # Don't suppress exceptions

In [ ]:
# DEMO: Using Timer context manager

print("Normal execution:")
with Timer() as t:
    time.sleep(0.1)
    print("   Processing...")

print(f"   Elapsed accessible: {t.elapsed:.4f}s")

print("\nWith exception:")
try:
    with Timer() as t:
        time.sleep(0.05)
        raise ValueError("Something went wrong!")
except ValueError:
    print("   Caught the exception")
    print("   ✅ Timer still reported elapsed time!")

In [ ]:
# GENERATOR-BASED CONTEXT MANAGER (simpler syntax)

@contextmanager
def database_connection(connection_string: str):
    """
    Context manager using @contextmanager decorator.
    
    HOW IT WORKS:
    1. Code BEFORE yield = __enter__
    2. yield value = what 'as' variable gets
    3. Code AFTER yield = __exit__
    """
    # SETUP (like __enter__)
    print(f"🔌 Connecting to: {connection_string}")
    connection = {"status": "connected", "url": connection_string}
    
    try:
        yield connection  # Return connection to 'with' block
    finally:
        # CLEANUP (like __exit__) - ALWAYS runs
        print(f"🔌 Disconnecting from: {connection_string}")
        connection["status"] = "disconnected"

In [ ]:
# DEMO: Generator-based context manager

print("Using database connection:")
with database_connection("postgres://localhost/mydb") as conn:
    print(f"   Connection: {conn}")
    print("   Running queries...")

print(f"\nAfter 'with': {conn}")

---
## 5. Protocols - Duck Typing with Type Hints

### What is a Protocol?
- Defines expected **interface** without inheritance
- "If it has these methods, it's valid"
- Great for duck typing with type safety

In [ ]:
class Writer(Protocol):
    """
    PROTOCOL: Defines what a 'Writer' looks like.
    
    Any class with a write(str) -> None method is a Writer.
    No inheritance required!
    """
    def write(self, data: str) -> None:
        ...  # Ellipsis = method signature only


# These classes DON'T inherit from Writer
# But they ARE Writers because they have write() method

class FileWriter:
    def write(self, data: str) -> None:
        print(f"📄 Writing to file: {data}")


class S3Writer:
    def write(self, data: str) -> None:
        print(f"☁️ Writing to S3: {data}")


class ConsoleWriter:
    def write(self, data: str) -> None:
        print(f"🖥️ Console output: {data}")

In [ ]:
def save_data(writer: Writer, data: str) -> None:
    """
    Accepts ANY object with a write() method.
    
    Type checker knows writer has write() method.
    No inheritance required from the Writer protocol.
    """
    print(f"Saving: {data}")
    writer.write(data)
    print("Done!\n")

# All these work - they all satisfy the Writer protocol
save_data(FileWriter(), "report.csv")
save_data(S3Writer(), "backup.json")
save_data(ConsoleWriter(), "Hello, World!")

---
## 6. Testing with Pytest

### Key Pytest Concepts:
- **Fixtures**: Reusable setup/teardown
- **Parametrize**: Run same test with different inputs
- **Mocking**: Replace real objects with fakes

In [ ]:
# EXAMPLE: Pytest fixtures and tests (would be in test_*.py file)

"""
# conftest.py - shared fixtures
import pytest

@pytest.fixture
def sample_data():
    '''Fresh data for each test.'''
    return [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]

@pytest.fixture(scope="session")
def database():
    '''Expensive setup, reused across all tests.'''
    db = connect_to_test_db()
    yield db  # Tests run here
    db.close()  # Cleanup after all tests


# test_etl.py
def test_transform_uppercase(sample_data):
    '''Test that names are uppercased.'''
    result = transform(sample_data)
    assert result[0]["name"] == "ALICE"


@pytest.mark.parametrize("input,expected", [
    (2, 4),
    (3, 9),
    (4, 16),
])
def test_square(input, expected):
    '''Test with multiple inputs.'''
    assert input ** 2 == expected
"""

# DEMO: Simulate parametrized test
print("Simulating @pytest.mark.parametrize:")
test_cases = [(2, 4), (3, 9), (4, 16), (5, 25)]

for input_val, expected in test_cases:
    result = input_val ** 2
    status = "✅" if result == expected else "❌"
    print(f"  {status} square({input_val}) = {result}, expected {expected}")

In [ ]:
# MOCKING: Replace real dependencies with fakes

def fetch_user_from_api(user_id: int) -> dict:
    """In real code, this calls external API."""
    import requests
    response = requests.get(f"https://api.example.com/users/{user_id}")
    return response.json()


def get_user_name(user_id: int) -> str:
    """Gets user name from API."""
    user = fetch_user_from_api(user_id)
    return user["name"]


# TEST WITH MOCKING
print("Testing with mocked API:")

with patch('__main__.fetch_user_from_api') as mock_fetch:
    # Configure mock to return test data
    mock_fetch.return_value = {"id": 1, "name": "Test User"}
    
    # Call function under test
    result = get_user_name(1)
    
    # Verify
    print(f"  Result: {result}")
    print(f"  Mock was called: {mock_fetch.called}")
    print(f"  Called with: {mock_fetch.call_args}")
    
    assert result == "Test User"
    print("  ✅ Test passed!")

---
## 7. Advanced Type Hints

### Beyond Basic Types:
- `Optional[T]` = `T | None`
- `Union[A, B]` = either A or B
- `TypeVar` = generic placeholder
- `Literal` = specific values only

In [ ]:
from typing import Union, Literal

# OPTIONAL: Value or None
def find_user(user_id: int) -> Optional[dict]:
    """Returns user dict or None if not found."""
    users = {1: {"name": "Alice"}, 2: {"name": "Bob"}}
    return users.get(user_id)  # Returns None if not found


# UNION: Multiple possible types
def process_id(id_value: Union[str, int]) -> str:
    """Accepts either string or int ID."""
    return str(id_value).upper()


# LITERAL: Only specific values allowed
def set_log_level(level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]) -> None:
    """Only these 4 values are valid."""
    print(f"Log level set to: {level}")


# DEMO
print("Type hints demo:")
print(f"  find_user(1): {find_user(1)}")
print(f"  find_user(999): {find_user(999)}")
print(f"  process_id(123): {process_id(123)}")
print(f"  process_id('abc'): {process_id('abc')}")
set_log_level("DEBUG")

In [ ]:
# TYPEVAR: Generic type placeholder

T = TypeVar('T')  # T can be any type

def first_item(items: List[T]) -> T:
    """
    Returns first item from list.
    
    Type is preserved:
    - first_item([1, 2, 3]) -> int
    - first_item(["a", "b"]) -> str
    """
    return items[0]


def last_item(items: List[T]) -> T:
    """Returns last item from list."""
    return items[-1]


# DEMO
print("Generic functions:")
numbers = [1, 2, 3, 4, 5]
words = ["apple", "banana", "cherry"]

print(f"  first_item({numbers}) = {first_item(numbers)}")
print(f"  last_item({words}) = {last_item(words)}")

---
## Summary: Advanced Python Checklist

| Topic | Key Points |
|-------|------------|
| Decorators | `@functools.wraps`, factory for args |
| Exceptions | Create hierarchy, include context |
| Context Managers | `__enter__/__exit__` or `@contextmanager` |
| Protocols | Duck typing with type safety |
| Pytest | Fixtures, parametrize, mock |
| Type Hints | Optional, Union, TypeVar, Literal |